# ApexPINN: remaining M2/mid folds, one job at a time

Prepared, NOT RUN. The user launches this notebook on hosted Colab L4. Default: F4 only.
After exporting and reviewing F4, change NEXT_FOLD to F5, then F6 in separate executions.
Each fold has its own persistent run identity and unchanged T13 v1 budget: 20 tuning trials,
five inner calibration fits and five final members, 1000 updates each. No clean-pilot refit,
additional search, M1 ablation or restart-oracle fitting is launched here.

Review the figures before spending more compute. The full T13 matrix remains incomplete;
prioritizing these three folds does not remove its remaining requirements or establish G2/P2.
The completed clean-calibration pilot and original F1-F3 records stay unchanged.


In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive, files
except ImportError as exc:
    raise RuntimeError('Use a hosted Colab GPU. Local execution is prohibited.') from exc
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('A hosted Colab runtime is required.')
drive.mount('/content/drive')
os.environ['APEXPINN_REMOTE_RUNTIME'] = 'colab'


## 2. Choose one fold and refresh your budget estimate

Edit NEXT_FOLD to exactly one of F4/F5/F6. Re-enter your actual current compute-unit balance
before each new fold. Use only one runtime for a given run ID. Never run another copy because
the browser timer is stale. An existing completed job is reused instead of fitted again.

Your reported balance/rate is 100 units at 1.94 units/hour, or about 51.55 allocated L4 hours.
The editable 20-unit reserve leaves about 41.24 hours. Assumed per-fold durations below are
planning scenarios, not measured bounds or confidence intervals:

| Hours/fold | Three-fold hours | Units | Units left from 100 |
| --- | --- | --- | --- |
| 8 | 24 | 46.56 | 53.44 |
| 12 | 36 | 69.84 | 30.16 |
| 16 | 48 | 93.12 | 6.88 |

F2 v1 recorded 9.11 process hours. F3's 5.61-hour successful attempt excludes its earlier attempt;
the clean pilot's 10.58-hour successful attempt uses a different split and 25 fits. Neither is a
whole-study upper bound for a new fold. Setup, compilation, interruptions and idle allocated time
can add cost. This budget check cannot enforce a billing cap or guarantee credits are sufficient.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/apexpinn')
INPUT_BUNDLE = DRIVE_ROOT / 'inputs/apexpinn_t13_input_v1.zip'
BUNDLE_SHA256 = 'b787682f95e30b303bfe7a41a2739dd40051a200724f079ffcab84a7d5618a03'
NEXT_FOLD = 'F4'
if NEXT_FOLD not in ('F4', 'F5', 'F6'):
    raise ValueError('Select one remaining fold: F4, F5 or F6')
RUN_ID = f't13_m2_{NEXT_FOLD.lower()}_mid_run01'
AVAILABLE_UNITS = 100.0  # Refresh from your account before EACH new fold.
UNITS_PER_HOUR = 1.94
RESERVE_UNITS = 20.0    # Editable operating preference; not a research parameter.
PLANNING_HOURS_PER_JOB = 12.0  # Assumed; not a measured upper bound.
if not (AVAILABLE_UNITS >= 0 and UNITS_PER_HOUR > 0 and RESERVE_UNITS >= 0
        and PLANNING_HOURS_PER_JOB > 0):
    raise ValueError('Check budget inputs')
print('One selected job:', RUN_ID)
print('Planning cost:', PLANNING_HOURS_PER_JOB * UNITS_PER_HOUR, 'units')
print('Above-reserve hours:', (AVAILABLE_UNITS - RESERVE_UNITS) / UNITS_PER_HOUR)
EXPORT_TAG = 'checkpoint_01'

MODELS = ['M2']
FOLDS = [NEXT_FOLD]
PHIS = ['mid']
VARIANTS = ['main']
if len(BUNDLE_SHA256) != 64 or any(c not in '0123456789abcdef' for c in BUNDLE_SHA256):
    raise ValueError('Paste the archive SHA-256 from the local input receipt.')
if any(not value or any(c not in
       'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in value)
       for value in (RUN_ID, EXPORT_TAG)):
    raise ValueError('Run ID and export tag must contain only letters, digits, underscore or hyphen.')
RUN_ROOT = DRIVE_ROOT / 'runs' / RUN_ID

OVERLAY_BUNDLE = DRIVE_ROOT / 'inputs/apexpinn_t13_code_update_v4.zip'
OVERLAY_SHA256 = '0b6d9d66c3ba99d7c8986d0c143e8791bf611a518194ac58284e58c383e0ed4a'


## 3. Verify and restore the input to the remote VM

Large input reads happen on the VM after one verified copy from Drive. Checkpoints and results are
written directly to the persistent run folder on Drive. The bootstrap imports only standard-library
transport code from the already hash-verified ZIP, before installing the training environment.

### Restore the input from Drive upload parts

The connector limits each file to 100 MiB. These two parts contain the unchanged original ZIP. This cell checks every part and the full archive SHA, then writes the ZIP on the remote VM. It performs file I/O only. No model is fitted. The original notebook and bundle are preserved.

In [ ]:
import hashlib
import json
import re
import tempfile

def assemble_input_bundle(input_bundle, expected_sha, destination_directory):
    manifest = json.loads(Path(str(input_bundle) + '.parts.json').read_text())
    if manifest['archive_sha256'] != expected_sha:
        raise ValueError('Part manifest differs from the approved input receipt.')
    checksum = hashlib.sha256()
    with tempfile.NamedTemporaryFile(
        mode='wb', prefix='apexpinn_input_', suffix='.zip',
        dir=destination_directory, delete=False
    ) as output:
        assembled = Path(output.name)
        for part in manifest['parts']:
            if not re.fullmatch(re.escape(input_bundle.name) + r'\.part[0-9]{2}', part['name']):
                raise ValueError('Unsafe input part name.')
            part_hash = hashlib.sha256()
            count = 0
            with (input_bundle.parent / part['name']).open('rb') as stream:
                for block in iter(lambda: stream.read(1 << 20), b''):
                    output.write(block)
                    checksum.update(block)
                    part_hash.update(block)
                    count += len(block)
            if count != part['bytes'] or part_hash.hexdigest() != part['sha256']:
                raise ValueError('Input part hash or size differs.')
    if checksum.hexdigest() != expected_sha:
        raise ValueError('Assembled archive differs from the approved input SHA-256.')
    return assembled

if not INPUT_BUNDLE.exists():
    INPUT_BUNDLE = assemble_input_bundle(INPUT_BUNDLE, BUNDLE_SHA256, Path('/content'))
    print('Reassembled and hash-verified input ZIP on the remote VM:', INPUT_BUNDLE)


In [ ]:
import hashlib
import json

def file_sha(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if file_sha(INPUT_BUNDLE) != BUNDLE_SHA256:
    raise ValueError('Input archive hash mismatch.')
sys.path.insert(0, str(INPUT_BUNDLE) + '/src')
from apexpinn.cloud.transfer import unpack
import importlib.util
from apexpinn.cloud.transfer import unpack

if file_sha(OVERLAY_BUNDLE) != OVERLAY_SHA256:
    raise ValueError('Update archive hash mismatch')
OVERLAY_ROOT = Path('/content') / ('apexpinn_update_' + OVERLAY_SHA256[:12])
if not OVERLAY_ROOT.exists():
    unpack(OVERLAY_BUNDLE, OVERLAY_ROOT, expected_sha256=OVERLAY_SHA256, kind='input')
else:
    import zipfile
    with zipfile.ZipFile(OVERLAY_BUNDLE) as update:
        expected_overlay = json.loads(update.read('bundle_manifest.json'))
    if json.loads((OVERLAY_ROOT / 'bundle_manifest.json').read_text()) != expected_overlay:
        raise ValueError('Update manifest changed')
    for name, item in expected_overlay['files'].items():
        if file_sha(OVERLAY_ROOT / name) != item['sha256']:
            raise ValueError('Update workspace changed')
spec = importlib.util.spec_from_file_location(
    'apexpinn_handoff', OVERLAY_ROOT / 'src/apexpinn/cloud/handoff.py')
handoff = importlib.util.module_from_spec(spec)
spec.loader.exec_module(handoff)
WORKSPACE = Path('/content') / ('apexpinn_' + BUNDLE_SHA256[:8] + '_' + OVERLAY_SHA256[:12])
handoff.restore_overlay(INPUT_BUNDLE, OVERLAY_ROOT, WORKSPACE, base_sha256=BUNDLE_SHA256)
print('Verified updated workspace:', WORKSPACE)


## 4. Install locked Python 3.12 and CUDA dependencies

The Colab notebook kernel may have another Python version. Training uses the isolated project Python
3.12 environment. `uv sync --frozen` uses the supplied lockfile; it does not upgrade the research setup.
If dependency/device setup fails, stop and retain the error. No CPU training fallback is provided.

In [ ]:
import shutil
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
UV = shutil.which('uv')
subprocess.run(
    [UV, 'sync', '--frozen', '--extra', 'cuda', '--no-dev', '--python', '3.12'],
    cwd=WORKSPACE, check=True)
PYTHON = WORKSPACE / '.venv/bin/python'
subprocess.run([str(PYTHON), '-c',
    'import jax; jax.config.update("jax_enable_x64", True); '
    'from apexpinn.train.cloud_checkpoint import require_colab_gpu; '
    'require_colab_gpu(); print(jax.devices(), jax.config.x64_enabled)'], cwd=WORKSPACE, check=True)

subprocess.run([str(PYTHON), '-c',
    'import jax; '
    'assert all("L4" in d.device_kind for d in jax.devices()), "Select a hosted L4 runtime"'],
    cwd=WORKSPACE, check=True)


## 5. Inspect the job plan

This cell does not train. The main models have 20 unique inner-validation trials per fold/phi.
Five fixed-configuration calibration fits estimate member scales on inner validation, followed by
five separate outer-training fits. Calibration fits are explicitly timed and are not hidden tuning trials.
M1 ablations reuse main hyperparameters without another search.
M2 remains the user-selected forecasting engine, with M1 as comparator.

In [ ]:
command = [str(PYTHON), 'scripts/run_models.py', '--out', str(RUN_ROOT)]
for flag, values in [('--model', MODELS), ('--fold', FOLDS), ('--phi', PHIS), ('--variant', VARIANTS)]:
    for value in values:
        command += [flag, value]
RECOVERED_REVIEW = WORKSPACE / 'recovered_evaluations/M2_F1_mid_main/review.json'
command += ['--recovered-review', str(RECOVERED_REVIEW)]
subprocess.run(command + ['--plan'], cwd=WORKSPACE, check=True)


## 6. Verify the existing source-matched restart receipt

The v4 source already passed the M1/M2 restart oracle on the user's L4 runtime.
This cell verifies that receipt and every referenced source hash. It performs no optimizer updates.
If the receipt or code identity differs, stop and inspect the setup.


In [ ]:
SELFTEST_ROOT = DRIVE_ROOT / 'checks/t13_m1_f1_mid_patch01_0b6d9d66c3ba'
receipt = json.loads((SELFTEST_ROOT / 'selftest.json').read_text())
if receipt['schema'] != 'apexpinn.cloud_restart_oracle/v1':
    raise ValueError('Unexpected restart receipt schema')
for name in ['M1', 'M2']:
    if receipt['results'][name] != 'PASS: model, offsets, optimizer, RNG and continuation state match':
        raise ValueError('Original restart parity was not established')
manifest = json.loads((WORKSPACE / 'workspace_manifest.json').read_text())
sources = {name: item['sha256'] for name, item in manifest['files'].items()
           if name.startswith(('src/apexpinn/', 'scripts/')) and name.endswith('.py')}
if receipt['instrument']['source_sha256'] != sources:
    raise ValueError('Restart receipt belongs to different source')
for name, expected in sources.items():
    if file_sha(WORKSPACE / name) != expected:
        raise ValueError('Source changed after restore: ' + name)
print('Verified existing restart parity and v4 source. No restart fits were run.')


## 7. Launch exactly the selected fold

This starts real training. Check the plan above: M2, selected F4/F5/F6, mid, main only.
The numerical engine and v4 source stay unchanged. The existing source-matched restart receipt
is reused; no extra restart fits. A same-VM nonblocking process lock prevents another launch
for the same run ID while the first process still holds it. It does not coordinate different VMs.

Checkpoints and process records persist on Drive. Observe progress.json and checkpoint steps;
the Colab cell timer is not a reliable elapsed-time source. Open .log files can appear late through
the Drive API. Resume with identical source/data/environment/run identity after an interruption;
do not change trial budgets, treat completed slots as fresh, or overwrite old exports.


In [ ]:
JOB = RUN_ROOT / 'jobs' / f'M2_{NEXT_FOLD}_mid_main' / 'job.json'
if JOB.is_file():
    print('Completed selected job already exists. Continue to verification/export; no fitting launched.')
else:
    if AVAILABLE_UNITS - RESERVE_UNITS < PLANNING_HOURS_PER_JOB * UNITS_PER_HOUR:
        raise RuntimeError('Planning cost exceeds units above reserve. Review balance before starting.')
    FLOCK = shutil.which('flock')
    if not FLOCK:
        raise RuntimeError('Hosted Linux flock is required for same-VM duplicate-launch protection.')
    LOG_ROOT = DRIVE_ROOT / 'logs' / RUN_ID / 'training'
    lock = Path('/content') / (RUN_ID + '.launch.lock')
    subprocess.run([FLOCK, '--nonblock', str(lock), str(PYTHON), '-m', 'apexpinn.cloud.process',
                    '--cwd', str(WORKSPACE), '--logs', str(LOG_ROOT), '--', *command], check=True)


## 8. Read saved status and export results

The result folder contains declarations, source snapshots, trial ledgers, calibration/final fits,
checkpoints, forecasts, metrics, S3/S4 diagnostics and a report. A partial shard reports G2 NOT ESTABLISHED.
G2 acceptance and user-owned P2 require a separate review of the complete study.

You can rerun the export after an interrupted job because committed checkpoints already live on Drive.
Use a new export tag for each snapshot; existing ZIPs are never overwritten.

In [ ]:
JOB = RUN_ROOT / 'jobs' / f'M2_{NEXT_FOLD}_mid_main' / 'job.json'
if not JOB.is_file():
    raise RuntimeError('Selected job incomplete. Inspect its persistent logs/checkpoints.')
job = json.loads(JOB.read_text())
declaration = json.loads((RUN_ROOT / 'declaration.json').read_text())
if (job['run_identity'] != declaration['identity']
        or job['conditions']['model'] != 'M2' or job['population']['fold'] != NEXT_FOLD
        or job['conditions']['phi'] != 0.06 or job['conditions']['variant'] != 'main'
        or len(job['trials']) != 20 or len(job['calibration_fits']) != 5 or len(job['final_fits']) != 5):
    raise ValueError('Completed selected job contract differs')
for name, expected in job['files'].items():
    if file_sha(JOB.parent / name) != expected:
        raise ValueError('Saved job artifact changed: ' + name)
report = RUN_ROOT / 'eval_physics.md'
if report.exists():
    print(report.read_text()[:6000])
else:
    print('No completed report yet. Inspect jobs/*/progress.json in the persistent run folder.')
EXPORT = DRIVE_ROOT / 'exports' / (RUN_ID + '_' + EXPORT_TAG + '.zip')
EXPORT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [str(PYTHON), '-m', 'apexpinn.cloud.transfer', 'pack-results',
     '--root', str(RUN_ROOT), '--out', str(EXPORT)], cwd=WORKSPACE, check=True)
print('Result archive:', EXPORT)
print('Receipt:', str(EXPORT) + '.receipt.json')
print('SHA-256:', file_sha(EXPORT))


## 9. Export, review, then release the runtime

Return the versioned ZIP and receipt for verification. This per-shard report may include recovered
M2/F1 again; do not sum its collector count with earlier exports. Training completion is not G2.
After the export is safely on Drive, use Colab's Disconnect and delete runtime when finished to avoid
idle credit use. No automatic runtime allocation/release or next-fold launch is performed.

Only after reviewing the previous result and your actual remaining balance, select the next fold
in the settings. The persistent output is unique to each fold. Increment EXPORT_TAG for another
export of the same run; never overwrite old evidence. Full T13/H2/G2/P2 and later stages remain open.
